# 07 modelos 15 a 17



## Modelo 15 — Profundidad de los árboles

**Qué probamos.** Entrenamos árboles de decisión con distintas profundidades y mínimos de muestras por hoja. También comparamos grupos de características y estrategias de selección.

**Qué pasó.** Entre las configuraciones evaluadas destacó una profundidad de 6 y un mínimo de 10 muestras por hoja. Su ganancia promedio quedó por debajo de la regresión logística de referencia en esa validación.

**Qué decidimos.** No utilizamos el árbol como modelo final. Además, al probar muchas combinaciones sobre los mismos meses, una diferencia pequeña podía depender de las solicitudes concretas que entraban en la cartera.

In [ ]:

import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score

# usamos las caracteristicas que ya tenemos en el colab
datos_m15 = train_fe.copy()
datos_m15["Fecha_Solicitud"] = pd.to_datetime(
    datos_m15["Fecha_Solicitud"]
)

columnas_m15 = {
    "V4": list(columnas_v4),
    "V4 + Motivo": list(columnas_v4) + ["Motivo_Prestamo"]
}

# definimos las tres campañas de validacion
meses_m15 = ["2025-10", "2025-11", "2025-12"]

# seleccionamos prestamos segun el valor esperado
def evaluar_m15(datos, probabilidades, estrategia):
    cartera = datos[
        ["Monto_Solicitado", "Tasa_Interes", "Estado_Final"]
    ].copy()

    cartera["Prob_Pago"] = probabilidades

    # calculamos el valor esperado y la rentabilidad por dolar
    cartera["EV"] = (
        cartera["Monto_Solicitado"] *
        (
            cartera["Prob_Pago"] * cartera["Tasa_Interes"]
            - (1 - cartera["Prob_Pago"])
        )
    )

    cartera["EV_Dolar"] = (
        cartera["EV"] / cartera["Monto_Solicitado"]
    )

    # ordenamos segun la estrategia elegida
    criterio = "EV" if estrategia == "EV" else "EV_Dolar"
    cartera = cartera.sort_values(criterio, ascending=False)

    # aprobamos prestamos sin superar el presupuesto
    capital = 0
    seleccionados = []

    for fila in cartera.itertuples():
        if fila.EV <= 0:
            continue

        if capital + fila.Monto_Solicitado <= 2_000_000:
            seleccionados.append(fila.Index)
            capital += fila.Monto_Solicitado

    aprobados = cartera.loc[seleccionados]

    # calculamos las ganancias reales de los prestamos aprobados
    ganancias = np.where(
        aprobados["Estado_Final"] == 1,
        aprobados["Monto_Solicitado"] * aprobados["Tasa_Interes"],
        -aprobados["Monto_Solicitado"]
    )

    return {
        "Ganancia": round(float(ganancias.sum()), 2),
        "Capital": float(capital),
        "Aprobados": len(aprobados),
        "Defaults": int((aprobados["Estado_Final"] == 0).sum())
    }


In [ ]:

resultados_m15 = []

profundidades = [2, 3, 4, 5, 6, 7, 8, 10, 12, 15]
minimos_hoja = [10, 30, 50, 100]

# recorremos los dos conjuntos de caracteristicas
for nombre_features, columnas in columnas_m15.items():

    # probamos distintas profundidades y tamaños de hoja
    for profundidad in profundidades:
        for minimo in minimos_hoja:

            # evaluamos cada mes entrenando solo con meses anteriores
            for mes in meses_m15:
                fecha_inicio = pd.Timestamp(mes + "-01")
                fecha_fin = fecha_inicio + pd.offsets.MonthBegin(1)

                entrenamiento = datos_m15[
                    datos_m15["Fecha_Solicitud"] < fecha_inicio
                ]

                validacion = datos_m15[
                    (datos_m15["Fecha_Solicitud"] >= fecha_inicio) &
                    (datos_m15["Fecha_Solicitud"] < fecha_fin) &
                    (datos_m15["Monto_Solicitado"] <= 10900)
                ]

                # agregamos one hot encoding cuando usamos el motivo
                if nombre_features == "V4 + Motivo":
                    preprocesador = ColumnTransformer([
                        ("numericas", "passthrough", columnas_v4),
                        ("motivo", OneHotEncoder(
                            handle_unknown="ignore"
                        ), ["Motivo_Prestamo"])
                    ])

                    modelo = Pipeline([
                        ("preprocesador", preprocesador),
                        ("arbol", DecisionTreeClassifier(
                            max_depth=profundidad,
                            min_samples_leaf=minimo,
                            random_state=42
                        ))
                    ])
                else:
                    modelo = DecisionTreeClassifier(
                        max_depth=profundidad,
                        min_samples_leaf=minimo,
                        random_state=42
                    )

                # entrenamos y calculamos probabilidades
                modelo.fit(
                    entrenamiento[columnas],
                    entrenamiento["Estado_Final"]
                )

                probabilidades = modelo.predict_proba(
                    validacion[columnas]
                )[:, 1]

                accuracy = accuracy_score(
                    validacion["Estado_Final"],
                    modelo.predict(validacion[columnas])
                )

                # probamos las dos formas de seleccionar prestamos
                for estrategia in ["EV", "EV_Dolar"]:
                    resultado = evaluar_m15(
                        validacion,
                        probabilidades,
                        estrategia
                    )

                    resultados_m15.append({
                        "Features": nombre_features,
                        "Profundidad": profundidad,
                        "Min_Hoja": minimo,
                        "Mes": mes,
                        "Estrategia": estrategia,
                        "Accuracy": accuracy,
                        **resultado
                    })

            print(
                f"Terminado: {nombre_features} - "
                f"profundidad {profundidad} - hoja {minimo}"
            )

# guardamos los resultados de todas las pruebas
tabla_m15 = pd.DataFrame(resultados_m15)

print("\nEvaluaciones realizadas:", len(tabla_m15))


In [ ]:

# resumimos los resultados de las tres campañas
resumen_m15 = tabla_m15.groupby(
    ["Features", "Profundidad", "Min_Hoja", "Estrategia"]
).agg(
    Ganancia_Promedio=("Ganancia", "mean"),
    Peor_Mes=("Ganancia", "min"),
    Mejor_Mes=("Ganancia", "max"),
    Accuracy_Promedio=("Accuracy", "mean"),
    Defaults_Promedio=("Defaults", "mean")
).reset_index()

# mostramos las diez mejores configuraciones
resumen_m15 = resumen_m15.sort_values(
    ["Ganancia_Promedio", "Peor_Mes"],
    ascending=False
)

print("TOP 10 CONFIGURACIONES")
display(resumen_m15.head(10).round(2))

# comparamos el mejor arbol contra la regresion de nuestros compañeros
mejor_m15 = resumen_m15.iloc[0]

print("\nMEJOR ARBOL")
print("Features:", mejor_m15["Features"])
print("Profundidad:", mejor_m15["Profundidad"])
print("Minimo por hoja:", mejor_m15["Min_Hoja"])
print("Seleccion:", mejor_m15["Estrategia"])
print("Ganancia promedio:", round(mejor_m15["Ganancia_Promedio"], 2))
print("Peor mes:", round(mejor_m15["Peor_Mes"], 2))

print(
    "Diferencia contra regresion + motivo:",
    round(mejor_m15["Ganancia_Promedio"] - 509955.80, 2)
)


In [ ]:

# buscamos los resultados mensuales de la mejor configuracion
detalle_m15 = tabla_m15[
    (tabla_m15["Features"] == mejor_m15["Features"]) &
    (tabla_m15["Profundidad"] == mejor_m15["Profundidad"]) &
    (tabla_m15["Min_Hoja"] == mejor_m15["Min_Hoja"]) &
    (tabla_m15["Estrategia"] == mejor_m15["Estrategia"])
]

display(
    detalle_m15[
        ["Mes", "Ganancia", "Capital", "Aprobados", "Defaults", "Accuracy"]
    ].sort_values("Mes").round(2)
)

# guardamos todas las pruebas para el informe
tabla_m15.to_csv(
    "resultados_modelo15.csv",
    index=False
)

resumen_m15.to_csv(
    "resumen_modelo15.csv",
    index=False
)


## Modelo 16 — Combinación de modelos (ensembles)

**Qué probamos.** Combinamos probabilidades de regresión logística, Random Forest y Gradient Boosting, utilizando distintos pesos.

**Qué pasó.** La regresión logística sola obtuvo USD 509.955,80 de ganancia mensual promedio en la validación utilizada. La mezcla de 80% logística y 20% Gradient Boosting obtuvo USD 500.463,68. El ensemble enviado a Kaggle quedó cerca de USD 126.000, también por debajo de nuestra mejor regresión de ese momento.

**Qué decidimos.** No conservar el ensemble. En estas pruebas, combinar modelos no nos ayudó a elegir préstamos más rentables.

In [ ]:

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# usamos las variables de v4 y el motivo del prestamo
numericas_m16 = list(columnas_v4)
columnas_m16 = numericas_m16 + ["Motivo_Prestamo"]

datos_m16 = train_fe.copy()
datos_m16["Fecha_Solicitud"] = pd.to_datetime(
    datos_m16["Fecha_Solicitud"]
)

meses_m16 = ["2025-10", "2025-11", "2025-12"]

# preparamos las variables para cada tipo de modelo
def crear_modelo_m16(tipo):
    if tipo == "Logistica":
        preprocesador = ColumnTransformer([
            ("numericas", StandardScaler(), numericas_m16),
            ("motivo", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ), ["Motivo_Prestamo"])
        ])

        clasificador = LogisticRegression(
            max_iter=1000,
            random_state=42
        )

    else:
        preprocesador = ColumnTransformer([
            ("numericas", "passthrough", numericas_m16),
            ("motivo", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ), ["Motivo_Prestamo"])
        ])

        if tipo == "Random Forest":
            clasificador = RandomForestClassifier(
                n_estimators=200,
                max_depth=10,
                min_samples_leaf=10,
                random_state=42,
                n_jobs=-1
            )

        elif tipo == "Gradient Boosting":
            clasificador = GradientBoostingClassifier(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=3,
                min_samples_leaf=10,
                random_state=42
            )

        else:
            raise ValueError("Modelo no reconocido")

    return Pipeline([
        ("preprocesador", preprocesador),
        ("modelo", clasificador)
    ])


# seleccionamos prestamos por ganancia esperada por dolar
def evaluar_cartera_m16(datos, probabilidades):
    cartera = datos[
        ["Monto_Solicitado", "Tasa_Interes", "Estado_Final"]
    ].copy()

    cartera["Prob_Pago"] = probabilidades

    # calculamos la rentabilidad esperada
    cartera["EV"] = cartera["Monto_Solicitado"] * (
        cartera["Prob_Pago"] * cartera["Tasa_Interes"]
        - (1 - cartera["Prob_Pago"])
    )

    cartera["EV_Dolar"] = (
        cartera["EV"] / cartera["Monto_Solicitado"]
    )

    cartera = cartera.sort_values(
        "EV_Dolar", ascending=False
    )

    # elegimos los prestamos respetando el presupuesto
    capital = 0
    aprobados = []

    for indice, fila in cartera.iterrows():
        monto = fila["Monto_Solicitado"]

        if fila["EV"] > 0 and capital + monto <= 2_000_000:
            aprobados.append(indice)
            capital += monto

    seleccionados = cartera.loc[aprobados]

    # calculamos la ganancia real de los aprobados
    ganancias = np.where(
        seleccionados["Estado_Final"] == 1,
        seleccionados["Monto_Solicitado"] *
        seleccionados["Tasa_Interes"],
        -seleccionados["Monto_Solicitado"]
    )

    return {
        "Ganancia": round(float(ganancias.sum()), 2),
        "Capital": capital,
        "Aprobados": len(seleccionados),
        "Defaults": int(
            (seleccionados["Estado_Final"] == 0).sum()
        )
    }


In [ ]:

# definimos las combinaciones de probabilidades
combinaciones_m16 = {
    "Logistica sola": (1.0, 0.0, 0.0),
    "Random Forest solo": (0.0, 1.0, 0.0),
    "Gradient Boosting solo": (0.0, 0.0, 1.0),
    "Logistica 80 + RF 20": (0.8, 0.2, 0.0),
    "Logistica 60 + RF 40": (0.6, 0.4, 0.0),
    "Logistica 80 + GB 20": (0.8, 0.0, 0.2),
    "Logistica 60 + GB 40": (0.6, 0.0, 0.4),
    "Logistica 60 + RF 20 + GB 20": (0.6, 0.2, 0.2),
    "Promedio de los tres": (1/3, 1/3, 1/3)
}

resultados_m16 = []

# entrenamos los tres modelos para cada mes
for mes in meses_m16:
    inicio = pd.Timestamp(mes + "-01")
    fin = inicio + pd.offsets.MonthBegin(1)

    entrenamiento = datos_m16[
        datos_m16["Fecha_Solicitud"] < inicio
    ]

    validacion = datos_m16[
        (datos_m16["Fecha_Solicitud"] >= inicio) &
        (datos_m16["Fecha_Solicitud"] < fin) &
        (datos_m16["Monto_Solicitado"] <= 10900)
    ]

    probabilidades_mes = {}

    for tipo in [
        "Logistica",
        "Random Forest",
        "Gradient Boosting"
    ]:
        modelo = crear_modelo_m16(tipo)

        modelo.fit(
            entrenamiento[columnas_m16],
            entrenamiento["Estado_Final"]
        )

        probabilidades_mes[tipo] = modelo.predict_proba(
            validacion[columnas_m16]
        )[:, 1]

    # mezclamos las probabilidades y evaluamos cada cartera
    for nombre, pesos in combinaciones_m16.items():
        p_log, p_rf, p_gb = pesos

        probabilidades = (
            p_log * probabilidades_mes["Logistica"]
            + p_rf * probabilidades_mes["Random Forest"]
            + p_gb * probabilidades_mes["Gradient Boosting"]
        )

        resultado = evaluar_cartera_m16(
            validacion, probabilidades
        )

        resultados_m16.append({
            "Combinacion": nombre,
            "Mes": mes,
            **resultado
        })

    print("Terminado:", mes)

tabla_m16 = pd.DataFrame(resultados_m16)

print("Evaluaciones realizadas:", len(tabla_m16))


In [ ]:

# resumimos la ganancia de cada combinacion
resumen_m16 = tabla_m16.groupby(
    "Combinacion"
).agg(
    Ganancia_Promedio=("Ganancia", "mean"),
    Peor_Mes=("Ganancia", "min"),
    Mejor_Mes=("Ganancia", "max"),
    Defaults_Promedio=("Defaults", "mean"),
    Aprobados_Promedio=("Aprobados", "mean")
).reset_index()

resumen_m16 = resumen_m16.sort_values(
    "Ganancia_Promedio", ascending=False
)

print("RESULTADOS DEL ENSEMBLE")
display(resumen_m16.round(2))

# mostramos la ganancia de cada mes por combinacion
tabla_mensual_m16 = tabla_m16.pivot(
    index="Combinacion",
    columns="Mes",
    values="Ganancia"
)

display(tabla_mensual_m16.round(2))

# vemos la diferencia con la mejor regresion anterior
mejor_m16 = resumen_m16.iloc[0]

print("\nMejor combinacion:", mejor_m16["Combinacion"])
print("Ganancia promedio:", round(
    mejor_m16["Ganancia_Promedio"], 2
))
print("Diferencia contra referencia:", round(
    mejor_m16["Ganancia_Promedio"] - 509955.80, 2
))


In [ ]:

# elegimos la mejor combinacion si supera a la logistica sola
ganancia_base_m16 = resumen_m16.set_index(
    "Combinacion"
).loc["Logistica sola", "Ganancia_Promedio"]

nombre_elegido_m16 = mejor_m16["Combinacion"]

if (
    nombre_elegido_m16 != "Logistica sola"
    and mejor_m16["Ganancia_Promedio"] > ganancia_base_m16
):

    pesos = combinaciones_m16[nombre_elegido_m16]
    prob_test_modelos = {}

    # entrenamos los modelos con todas las solicitudes historicas
    for tipo in [
        "Logistica",
        "Random Forest",
        "Gradient Boosting"
    ]:
        modelo = crear_modelo_m16(tipo)

        modelo.fit(
            train_fe[columnas_m16],
            train_fe["Estado_Final"]
        )

        prob_test_modelos[tipo] = modelo.predict_proba(
            test_fe[columnas_m16]
        )[:, 1]

    # calculamos la probabilidad final combinada
    prob_final_m16 = (
        pesos[0] * prob_test_modelos["Logistica"]
        + pesos[1] * prob_test_modelos["Random Forest"]
        + pesos[2] * prob_test_modelos["Gradient Boosting"]
    )

    # armamos la cartera y calculamos la rentabilidad esperada
    cartera_test_m16 = test_fe[
        ["ID_Solicitud", "Monto_Solicitado", "Tasa_Interes"]
    ].copy()

    cartera_test_m16["EV"] = (
        cartera_test_m16["Monto_Solicitado"] *
        (
            prob_final_m16 * cartera_test_m16["Tasa_Interes"]
            - (1 - prob_final_m16)
        )
    )

    cartera_test_m16["EV_Dolar"] = (
        cartera_test_m16["EV"] /
        cartera_test_m16["Monto_Solicitado"]
    )

    cartera_test_m16 = cartera_test_m16.sort_values(
        "EV_Dolar", ascending=False
    )

    # seleccionamos los prestamos hasta completar el presupuesto
    cartera_test_m16["Decision_Aprobacion"] = 0
    capital_m16 = 0

    for indice, fila in cartera_test_m16.iterrows():
        monto = fila["Monto_Solicitado"]

        if fila["EV"] > 0 and capital_m16 + monto <= 2_000_000:
            cartera_test_m16.loc[indice, "Decision_Aprobacion"] = 1
            capital_m16 += monto

    # dejamos el archivo en el orden original de kaggle
    submission_v5 = test[["ID_Solicitud"]].merge(
        cartera_test_m16[["ID_Solicitud", "Decision_Aprobacion"]],
        on="ID_Solicitud",
        how="left",
        validate="one_to_one"
    )

    # comprobamos el formato y el presupuesto
    assert len(submission_v5) == len(test)
    assert submission_v5["Decision_Aprobacion"].notna().all()
    assert submission_v5["Decision_Aprobacion"].isin([0, 1]).all()

    capital_real = (
        submission_v5.merge(
            test[["ID_Solicitud", "Monto_Solicitado"]],
            on="ID_Solicitud"
        ).eval("Decision_Aprobacion * Monto_Solicitado").sum()
    )

    assert capital_real <= 2_000_000

    # guardamos el archivo sin enviarlo a kaggle
    submission_v5.to_csv("submission_v5_ensemble.csv", index=False)

    print("Archivo preparado: submission_v5_ensemble.csv")
    print("Combinacion:", nombre_elegido_m16)
    print("Aprobados:", submission_v5["Decision_Aprobacion"].sum())
    print("Capital utilizado:", capital_real)

else:
    print("El ensemble no mejoro a la regresion sola.")
    print("No generamos una nueva submission.")


In [ ]:

import numpy as np

# entrenamos los dos modelos con todos los datos historicos
modelo_log_v5 = crear_modelo_m16("Logistica")
modelo_gb_v5 = crear_modelo_m16("Gradient Boosting")

modelo_log_v5.fit(
    train_fe[columnas_m16],
    train_fe["Estado_Final"]
)

modelo_gb_v5.fit(
    train_fe[columnas_m16],
    train_fe["Estado_Final"]
)

# calculamos las probabilidades de pago de cada modelo
prob_log_v5 = modelo_log_v5.predict_proba(
    test_fe[columnas_m16]
)[:, 1]

prob_gb_v5 = modelo_gb_v5.predict_proba(
    test_fe[columnas_m16]
)[:, 1]

# combinamos las probabilidades
prob_ensemble_v5 = (
    0.8 * prob_log_v5
    + 0.2 * prob_gb_v5
)

# calculamos el valor esperado de cada prestamo
cartera_ensemble_v5 = test_fe[
    ["ID_Solicitud", "Monto_Solicitado", "Tasa_Interes"]
].copy()

cartera_ensemble_v5["EV"] = (
    cartera_ensemble_v5["Monto_Solicitado"] *
    (
        prob_ensemble_v5 * cartera_ensemble_v5["Tasa_Interes"]
        - (1 - prob_ensemble_v5)
    )
)

cartera_ensemble_v5["EV_Dolar"] = (
    cartera_ensemble_v5["EV"] /
    cartera_ensemble_v5["Monto_Solicitado"]
)

# ordenamos los prestamos por rentabilidad esperada
cartera_ensemble_v5 = cartera_ensemble_v5.sort_values(
    "EV_Dolar", ascending=False
)

# aprobamos prestamos hasta completar el presupuesto
cartera_ensemble_v5["Decision_Aprobacion"] = 0
capital_ensemble_v5 = 0

for indice, fila in cartera_ensemble_v5.iterrows():
    monto = fila["Monto_Solicitado"]

    if fila["EV"] > 0 and capital_ensemble_v5 + monto <= 2_000_000:
        cartera_ensemble_v5.loc[
            indice, "Decision_Aprobacion"
        ] = 1

        capital_ensemble_v5 += monto

# recuperamos el orden original del test
submission_v5 = test[["ID_Solicitud"]].merge(
    cartera_ensemble_v5[["ID_Solicitud", "Decision_Aprobacion"]],
    on="ID_Solicitud",
    how="left",
    validate="one_to_one"
)

# comprobamos el presupuesto y el formato de kaggle
assert len(submission_v5) == len(test)
assert submission_v5["Decision_Aprobacion"].notna().all()
assert submission_v5["Decision_Aprobacion"].isin([0, 1]).all()

capital_verificado = (
    submission_v5
    .merge(test[["ID_Solicitud", "Monto_Solicitado"]], on="ID_Solicitud")
    .eval("Decision_Aprobacion * Monto_Solicitado")
    .sum()
)

assert capital_verificado <= 2_000_000

# guardamos y descargamos la submission
submission_v5.to_csv("submission_v5_ensemble.csv", index=False)

print("Prestamos aprobados:", submission_v5["Decision_Aprobacion"].sum())
print("Capital utilizado:", capital_verificado)
print("Presupuesto restante:", 2_000_000 - capital_verificado)

print("Archivo disponible para descargar:", "submission_v5_ensemble.csv")

In [ ]:

from sklearn.tree import DecisionTreeClassifier

import numpy as np
import pandas as pd

# entrenamos el mejor arbol de decision
arbol_v6 = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=10,
    random_state=42
)

arbol_v6.fit(train_fe[columnas_v4], train_fe["Estado_Final"])

# calculamos probabilidades y ganancias esperadas
prob_v6 = arbol_v6.predict_proba(test_fe[columnas_v4])[:, 1]

cartera_v6 = test_fe[
    ["ID_Solicitud", "Monto_Solicitado", "Tasa_Interes"]
].copy()

cartera_v6["EV_Dolar"] = (
    prob_v6 * (1 + cartera_v6["Tasa_Interes"].to_numpy()) - 1
)

cartera_v6 = cartera_v6.sort_values("EV_Dolar", ascending=False)
cartera_v6["Decision_Aprobacion"] = 0

# aprobamos sin superar los dos millones
capital = 0

for indice, fila in cartera_v6.iterrows():
    monto = fila["Monto_Solicitado"]

    if fila["EV_Dolar"] > 0 and capital + monto <= 2_000_000:
        cartera_v6.loc[indice, "Decision_Aprobacion"] = 1
        capital += monto

# generamos la submission en el orden original
submission_v6 = test[["ID_Solicitud"]].merge(
    cartera_v6[["ID_Solicitud", "Decision_Aprobacion"]],
    on="ID_Solicitud",
    how="left",
    validate="one_to_one"
)

assert len(submission_v6) == len(test)
assert submission_v6["Decision_Aprobacion"].notna().all()
assert capital <= 2_000_000

submission_v6.to_csv("submission_v6_arbol.csv", index=False)

print("Aprobados:", int(submission_v6["Decision_Aprobacion"].sum()))
print("Capital utilizado:", capital)

print("Archivo disponible para descargar:", "submission_v6_arbol.csv")

## Modelo 17 — Regresiones con menos variables

**Qué probamos.** Simplificamos la regresión logística para ver si se podía obtener una cartera parecida usando menos información. Comparamos conjuntos reducidos de 4, 7 y 8 variables contra la versión con más características.

**Qué pasó.** Ninguna simplificación mejoró la referencia en la evaluación mensual. Con 7 variables obtuvimos USD 499.363,56 de promedio; con 8, USD 492.744,53. La versión de 4 variables cayó hasta aproximadamente USD 194.685.

**Qué decidimos.** Mantener las características que veníamos utilizando. Reducir el modelo demasiado hizo perder información útil para esta tarea.

In [ ]:

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# definimos las variables de cada modelo
features_m17 = {
    "Basico": [
        "Monto_Solicitado",
        "Tasa_Interes",
        "Ingreso_Mensual",
        "Ratio_Monto_Ingreso"
    ],

    "Basico + Atrasos": [
        "Monto_Solicitado",
        "Tasa_Interes",
        "Ingreso_Mensual",
        "Ratio_Monto_Ingreso",
        "Atraso_Promedio",
        "Atraso_Maximo",
        "Frecuencia_Atraso"
    ],

    "Basico + Bancario + Motivo": [
        "Monto_Solicitado",
        "Tasa_Interes",
        "Ingreso_Mensual",
        "Ratio_Monto_Ingreso",
        "Saldo_Promedio",
        "Atraso_Promedio",
        "Frecuencia_Atraso",
        "Motivo_Prestamo"
    ],

    "Modelo actual": list(columnas_m16)
}

# usamos las solicitudes que ya tenemos preparadas
datos_m17 = train_fe.copy()
datos_m17["Fecha_Solicitud"] = pd.to_datetime(
    datos_m17["Fecha_Solicitud"]
)

meses_m17 = ["2025-10", "2025-11", "2025-12"]


In [ ]:

# armamos una regresion logistica segun las variables elegidas
def crear_logistica_m17(columnas):
    numericas = [
        col for col in columnas
        if col != "Motivo_Prestamo"
    ]

    transformaciones = [
        ("numericas", StandardScaler(), numericas)
    ]

    # codificamos el motivo cuando forma parte del modelo
    if "Motivo_Prestamo" in columnas:
        transformaciones.append((
            "motivo",
            OneHotEncoder(handle_unknown="ignore"),
            ["Motivo_Prestamo"]
        ))

    return Pipeline([
        ("preprocesador", ColumnTransformer(transformaciones)),
        ("modelo", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])


resultados_m17 = []

# probamos cada configuracion en los tres meses
for nombre, columnas in features_m17.items():
    for mes in meses_m17:
        inicio = pd.Timestamp(mes + "-01")
        fin = inicio + pd.offsets.MonthBegin(1)

        # entrenamos solamente con los meses anteriores
        entrenamiento = datos_m17[
            datos_m17["Fecha_Solicitud"] < inicio
        ]

        validacion = datos_m17[
            (datos_m17["Fecha_Solicitud"] >= inicio) &
            (datos_m17["Fecha_Solicitud"] < fin) &
            (datos_m17["Monto_Solicitado"] <= 10900)
        ]

        modelo = crear_logistica_m17(columnas)
        modelo.fit(
            entrenamiento[columnas],
            entrenamiento["Estado_Final"]
        )

        # calculamos las probabilidades de pago
        probabilidades = modelo.predict_proba(
            validacion[columnas]
        )[:, 1]

        # evaluamos la cartera mensual completa
        resultado = evaluar_cartera_m16(
            validacion, probabilidades
        )

        resultados_m17.append({
            "Modelo": nombre,
            "Mes": mes,
            "Segmento": "Hasta 10900",
            **resultado
        })

        # evaluamos por separado los prestamos pequeños
        mascara_pequenos = (
            validacion["Monto_Solicitado"] <= 3000
        )

        pequenos = validacion.loc[mascara_pequenos]
        prob_pequenos = probabilidades[mascara_pequenos.to_numpy()]

        resultado_pequenos = evaluar_cartera_m16(
            pequenos, prob_pequenos
        )

        resultados_m17.append({
            "Modelo": nombre,
            "Mes": mes,
            "Segmento": "Hasta 3000",
            **resultado_pequenos
        })

    print("Terminado:", nombre)

tabla_m17 = pd.DataFrame(resultados_m17)


In [ ]:

# calculamos la ganancia promedio de cada modelo y segmento
resumen_m17 = tabla_m17.groupby(
    ["Modelo", "Segmento"]
).agg(
    Ganancia_Promedio=("Ganancia", "mean"),
    Peor_Mes=("Ganancia", "min"),
    Mejor_Mes=("Ganancia", "max"),
    Aprobados_Promedio=("Aprobados", "mean"),
    Defaults_Promedio=("Defaults", "mean"),
    Capital_Promedio=("Capital", "mean")
).reset_index()

# mostramos las configuraciones ordenadas por ganancia
display(
    resumen_m17.sort_values(
        ["Segmento", "Ganancia_Promedio"],
        ascending=[True, False]
    ).round(2)
)

# mostramos las ganancias mensuales por modelo
display(
    tabla_m17.pivot_table(
        index=["Modelo", "Segmento"],
        columns="Mes",
        values="Ganancia"
    ).round(2)
)


In [ ]:

# comparamos las versiones simplificadas contra el modelo actual
comparacion_m17 = resumen_m17[
    resumen_m17["Segmento"] == "Hasta 10900"
].sort_values("Ganancia_Promedio", ascending=False)

display(comparacion_m17.round(2))

mejor_m17 = comparacion_m17.iloc[0]

print("Mejor modelo:", mejor_m17["Modelo"])
print("Ganancia promedio:", round(
    mejor_m17["Ganancia_Promedio"], 2
))

print("Referencia anterior: 509955.80")
print("Diferencia:", round(
    mejor_m17["Ganancia_Promedio"] - 509955.80, 2
))
